In [ ]:
import pandas as pd
import folium
import geopandas as gpd
from folium.plugins import MarkerCluster, HeatMap

# Load airport dataset
df = pd.read_csv("airports-india.csv")
df.columns = df.columns.str.strip()
df = df.dropna(subset=["latitude", "longitude"])
df["latitude"] = pd.to_numeric(df["latitude"], errors="coerce")
df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")
df = df.dropna(subset=["latitude", "longitude"])
df = df.drop_duplicates(subset="name")

center = [22.5, 79]

# Load corrected India state boundaries
states = gpd.read_file("india-states-full.geojson")

# Create map
def make_map():
    return folium.Map(
        location=center,
        zoom_start=5,
        tiles="OpenStreetMap",
        control_scale=True
    )

# Add corrected state boundaries
def add_india_outline(m):
    folium.GeoJson(
        states.__geo_interface__,
        name="Corrected India Boundaries",
        style_function=lambda feature: {
            "fillOpacity": 0,
            "color": "black",
            "weight": 1.5
        }
    ).add_to(m)

    return m


# --------------------------------------------------
# 1. POINT MAP
# --------------------------------------------------
point_map = make_map()

for _, r in df.iterrows():
    folium.CircleMarker(
        location=[r["latitude"], r["longitude"]],
        radius=4,
        tooltip=r["name"],
        color="blue",
        fill=True,
        fill_opacity=0.8
    ).add_to(point_map)

add_india_outline(point_map)
point_map.save("exp7_point_map.html")


# --------------------------------------------------
# 2. CLUSTER MAP
# --------------------------------------------------
cluster_map = make_map()
cluster = MarkerCluster()

for _, r in df.iterrows():
    folium.Marker(
        location=[r["latitude"], r["longitude"]],
        tooltip=r["name"]
    ).add_to(cluster)

cluster.add_to(cluster_map)
add_india_outline(cluster_map)
cluster_map.save("exp7_cluster_map.html")


# --------------------------------------------------
# 3. HEATMAP
# --------------------------------------------------
heatmap = make_map()

HeatMap(
    df[["latitude", "longitude"]].values.tolist(),
    radius=15,
    blur=10,
    min_opacity=0.4
).add_to(heatmap)

add_india_outline(heatmap)
heatmap.save("exp7_heatmap.html")


# --------------------------------------------------
# 4. CHOROPLETH MAP
# --------------------------------------------------
choropleth_map = make_map()

counts = df.groupby("region").size().reset_index(name="count")

# Match state names to airport regions
states_map = states.merge(
    counts,
    left_on="NAME_1",
    right_on="region",
    how="left"
)

states_map["count"] = states_map["count"].fillna(0)

folium.GeoJson(
    states_map.__geo_interface__,
    style_function=lambda feature: {
        "fillColor": (
            "#800026" if (feature["properties"].get("count") or 0) > 10
            else "#BD0026" if (feature["properties"].get("count") or 0) > 5
            else "#FD8D3C" if (feature["properties"].get("count") or 0) > 0
            else "#EEEEEE"
        ),
        "color": "black",
        "weight": 0.7,
        "fillOpacity": 0.65
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["NAME_1", "count"],
        aliases=["State:", "Airports:"]
    )
).add_to(choropleth_map)

choropleth_map.save("exp7_choropleth_map.html")





# --------------------------------------------------
# OUTPUT
# --------------------------------------------------
print("All 5 maps generated successfully!")
print("1. exp7_point_map.html")
print("2. exp7_choropleth_map.html")
print("3. exp7_cluster_map.html")
print("4. exp7_heatmap.html")




All 5 maps generated successfully!
1. exp7_point_map.html
2. exp7_choropleth_map.html
3. exp7_cluster_map.html
4. exp7_heatmap.html
5. exp7_flow_map.html


In [3]:
# --------------------------------------------------
# 5. FLOW MAP: ILLUSTRATIVE ROUTES FROM DELHI, MUMBAI, BENGALURU
# --------------------------------------------------
flow_map = make_map()

# airport label -> name keywords to look for in df["name"] (first hit wins)
AIRPORTS = {
    "Delhi":      ["Indira Gandhi International Airport"],
    "Mumbai":     ["Chhatrapati Shivaji", "Mumbai Airport"],
    "Bengaluru":  ["Kempegowda", "Bengaluru International", "Bangalore"],
    "Chennai":    ["Chennai International Airport"],
    "Kolkata":    ["Netaji Subhas Chandra Bose"],
    "Hyderabad":  ["Rajiv Gandhi International Airport"],
    "Ahmedabad":  ["Sardar Vallabhbhai Patel International Airport"],
    "Amritsar":   ["Sri Guru Ram Dass Jee International Airport"],
    "Trivandrum": ["Trivandrum International Airport"],
    "Goa":        ["Dabolim", "Goa International"],
}

# hub -> (line color, destinations)  <- edit these by hand
ROUTES = {
    "Delhi":     ("red",    ["Mumbai", "Bengaluru", "Chennai", "Ahmedabad", "Amritsar", "Trivandrum"]),
    "Mumbai":    ("green",  ["Bengaluru", "Chennai", "Kolkata", "Hyderabad", "Goa"]),
    "Bengaluru": ("purple", ["Chennai", "Hyderabad", "Kolkata", "Trivandrum"]),
}

def find_airport(label):
    for kw in AIRPORTS[label]:
        m = df[df["name"].str.contains(kw, case=False, na=False, regex=False)]
        if not m.empty:
            return m.iloc[0]
    print(f"NOT FOUND: {label}")
    return None

drawn = set()

for hub, (color, dests) in ROUTES.items():
    src = find_airport(hub)
    if src is None:
        continue
    src_coords = [src["latitude"], src["longitude"]]

    folium.Marker(
        location=src_coords,
        tooltip=f"{hub} (hub): {src['name']}",
        icon=folium.Icon(color=color if color != "purple" else "purple", icon="plane")
    ).add_to(flow_map)

    for d in dests:
        pair = frozenset((hub, d))
        if pair in drawn:          # skip if already drawn from the other hub
            continue
        airport = find_airport(d)
        if airport is None:
            continue
        dest_coords = [airport["latitude"], airport["longitude"]]

        folium.PolyLine(
            locations=[src_coords, dest_coords],
            color=color,
            weight=2,
            opacity=0.8,
            tooltip=f"{hub} → {d}"
        ).add_to(flow_map)

        folium.CircleMarker(
            location=dest_coords,
            radius=4,
            tooltip=airport["name"],
            color="blue",
            fill=True
        ).add_to(flow_map)
        drawn.add(pair)

add_india_outline(flow_map)
flow_map.save("exp7_flow_map.html")

NOT FOUND: Kolkata
NOT FOUND: Kolkata
